# Modeling Baselines: WNBA Player Valuation Engine

This notebook establishes trivial, simple, and basic machine learning baselines for predicting WNBA player salaries based on on-court production metrics. 

### Objectives:
1. Load and integrate historical multi-year data (2021–2025).
2. Enforce structural chronological separation using the `WNBALeakageProofSplitter`.
3. Evaluate baseline models ranging from trivial heuristics to basic tree-based regressors.
4. Assess out-of-fold historical cross-validation performance using primary, secondary, and business-focused KPIs.

### Models Evaluated:
* **Trivial Baseline:** `DummyRegressor` (Predicts historical mean salary)
* **Simple Feature Baseline:** `LinearRegression` on a single feature (`pts`)
* **Linear Baselines:** `LinearRegression` (Ordinary Least Squares) and `Ridge` (L2 Regularization)
* **Basic Tree Baselines:** `DecisionTreeRegressor` and `RandomForestRegressor`


## 1. Setup & Imports

In [8]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Configurations
np.random.seed(26)


In [9]:
# Mapping dictionaries for structural cleaning
teammap = {
    "Atlanta Dream": "ATL", "Chicago Sky": "CHI", "Connecticut Sun": "CON",
    "Dallas Wings": "DAL", "Golden State Valkyries": "GSV", "Indiana Fever": "IND",
    "Las Vegas Aces": "LVA", "Los Angeles Sparks": "LAS", "Minnesota Lynx": "MIN",
    "New York Liberty": "NYL", "Phoenix Mercury": "PHO", "Seattle Storm": "SEA",
    "Washington Mystics": "WAS"
}

name_fix = {
    "Anastasiia Kosu": "Anastasiia Olairi Kosu", "Janelle Salau\u0308n": "Janelle Salaun",
    "Janelle Sala\u00c3\u00bcn": "Janelle Salaun", "Le\u00c3\u00aefla Lacan": "Leila Lacan", 
    "Le\u00efla Lacan": "Leila Lacan", "Luisa Geisels\u00c3\u00b6der": "Luisa Geiselsoder",
    "Luisa Geisels\u00f6der": "Luisa Geiselsoder", "Mamignan Tour\u00c3\u00a9": "Mamignan Touré", 
    "Mamignan Tour\u00e9": "Mamignan Touré", "Mari\u00c3\u00a8me Badiane": "Marième Badiane",
    "Mari\u00e8me Badiane": "Marième Badiane", "Te-Hina PaoPao": "Te-Hina Paopao", 
    "Sika Kon\u00c3\u00a9": "Sika Kone", "Sika Kon\u00e9": "Sika Kone"
}

def clean_col(name):
    text = str(name).strip().lower()
    text = re.sub(r"\b202\d\b", "", text) 
    text = text.strip()
    if "salary" in text:
        text = "salary"
    if "signing" in text:
        text = "signing"
        
    text = text.replace("%", "pct")
    text = re.sub(r"[^0-9a-z]+", "_", text)
    return text.strip("_")

def clean_df(df):
    df.columns = [clean_col(col) for col in df.columns]
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype(str).str.strip().replace({"": np.nan, "—": np.nan, "nan": np.nan})
    return df


In [10]:
# Multi-year chronological data loading pipeline
all_seasons = []
repo = Path.cwd().parent
data_dir = repo / "data" / "raw"

for year in range(2021, 2026):
    try:
        adv = pd.read_csv(data_dir / f"{year}_advanced.csv")
        per = pd.read_csv(data_dir / f"{year}_per_game.csv")
        tot = pd.read_csv(data_dir / f"{year}_totals.csv")
        sal = pd.read_csv(data_dir / f"salary_{year}.csv")
        teamadv = pd.read_csv(data_dir / f"{year}_advanced-team.csv")
        stand = pd.read_csv(data_dir / f"{year}_wnba_standings.csv")
        
        adv, per, tot = clean_df(adv), clean_df(per), clean_df(tot)
        sal, teamadv, stand = clean_df(sal), clean_df(teamadv), clean_df(stand)
        
        sal["salary"] = pd.to_numeric(sal["salary"], errors="coerce")
        
        teamadv["team"] = teamadv["team"].str.replace("*", "", regex=False).str.strip().map(teammap)
        stand["team_name"] = stand["team_name"].str.replace("*", "", regex=False).str.strip().map(teammap)
        
        teamdf = teamadv.merge(stand, left_on="team", right_on="team_name", how="left", suffixes=("_adv", "_stand"))
        
        playerdf = per.merge(adv, on=["player", "team", "pos", "g", "mp"], how="outer")
        playerdf = playerdf.merge(tot, on=["player", "team", "pos", "g", "mp", "gs"], how="outer")
        playerdf["player"] = playerdf["player"].replace(name_fix)
        
        year_df = sal.merge(playerdf, on="player", how="inner", suffixes=("_sal", ""))
        year_df = year_df.merge(teamdf, on="team", how="left")
        
        year_df = year_df.copy()
        year_df['year'] = year
        all_seasons.append(year_df)
        print(f"Successfully integrated and verified data for the {year} season.")
        
    except FileNotFoundError as e:
        print(f"Skipping year {year}: Continuous file matrix not found ({e.filename})")

final_df = pd.concat(all_seasons, ignore_index=True)
final_df = final_df.drop(columns=['dummy_x', 'dummy_y'], errors='ignore')
final_df = final_df.dropna(subset=["salary"]).reset_index(drop=True)

Successfully integrated and verified data for the 2021 season.
Successfully integrated and verified data for the 2022 season.
Successfully integrated and verified data for the 2023 season.
Successfully integrated and verified data for the 2024 season.
Successfully integrated and verified data for the 2025 season.


In [11]:
class WNBALeakageProofSplitter:
    def __init__(self, target_holdout_year=2025, year_col='year', target_col='salary'):
        self.target_holdout_year = target_holdout_year
        self.year_col = year_col
        self.target_col = target_col
        
        self.leakage_blacklist = [
            'dummy_x', 'dummy_y', 'signing_bonus', 'cap_pct', 'cap_percentage',
            'estimated_earnings', 'base_salary'
        ]

    def _clean_features(self, df):
        ignore_cols = [self.target_col] + self.leakage_blacklist
        feature_cols = [col for col in df.columns if col not in ignore_cols]
        return df[feature_cols], df[self.target_col]

    def get_final_holdout_split(self, df):
        if self.year_col not in df.columns:
            raise KeyError(f"The required chronological column '{self.year_col}' was not found in the dataframe.")
            
        historical_pool = df[df[self.year_col] < self.target_holdout_year].copy()
        holdout_pool = df[df[self.year_col] == self.target_holdout_year].copy()
        
        X_train_hist, y_train_hist = self._clean_features(historical_pool)
        X_test_2025, y_test_2025 = self._clean_features(holdout_pool)
        
        return X_train_hist, y_train_hist, X_test_2025, y_test_2025

    def generate_historical_cv_folds(self, X_train_hist, y_train_hist):
        years = X_train_hist[self.year_col].unique()
        sorted_years = sorted(years)
        
        if len(sorted_years) < 2:
            raise ValueError("Insufficient historical years to generate chronological CV folds.")

        X_train_hist = X_train_hist.reset_index(drop=True)
        
        for i in range(1, len(sorted_years)):
            train_years = sorted_years[:i]
            val_year = sorted_years[i]
            
            train_idx = X_train_hist[X_train_hist[self.year_col].isin(train_years)].index.tolist()
            val_idx = X_train_hist[X_train_hist[self.year_col] == val_year].index.tolist()
            
            yield np.array(train_idx), np.array(val_idx)

In [12]:
# Instantiate Splitter and execute data partitioning
splitter = WNBALeakageProofSplitter(target_holdout_year=2025, year_col='year', target_col='salary')
X_train_hist, y_train_hist, X_test_2025, y_test_2025 = splitter.get_final_holdout_split(final_df)

def run_defensive_unit_tests():
    print("Executing Splitting Unit Tests...")
    
    assert X_train_hist['year'].max() == 2024, "Fail: Training data contains years >= 2025"
    assert X_test_2025['year'].min() == 2025 and X_test_2025['year'].max() == 2025, "Fail: Holdout set has non-2025 records"
    print("Pass: 2025 isolation secure.")

    for blacklisted_col in splitter.leakage_blacklist:
        assert blacklisted_col not in X_train_hist.columns, f"Fail: Feature leakage detected, {blacklisted_col} left in train"
        assert blacklisted_col not in X_test_2025.columns, f"Fail: Feature leakage detected, {blacklisted_col} left in test"
    print("Pass: Feature leakage columns purged completely.")

    fold_generator = splitter.generate_historical_cv_folds(X_train_hist, y_train_hist)
    for fold, (train_idx, val_idx) in enumerate(fold_generator, start=1):
        max_fold_train_year = X_train_hist.iloc[train_idx]['year'].max()
        min_fold_val_year = X_train_hist.iloc[val_idx]['year'].min()
        assert max_fold_train_year < min_fold_val_year, f"Fail: Temporal leakage in CV Fold {fold}"
        print(f"Pass: CV Fold {fold} chronological sequencing verified.")

    print("\nData structures secure for modeling.")

run_defensive_unit_tests()

Executing Splitting Unit Tests...
Pass: 2025 isolation secure.
Pass: Feature leakage columns purged completely.
Pass: CV Fold 1 chronological sequencing verified.
Pass: CV Fold 2 chronological sequencing verified.
Pass: CV Fold 3 chronological sequencing verified.

Data structures secure for modeling.


In [13]:
# Preprocessing Pipeline Configurations
# Drop non-predictive tracking keys from structural pipeline mapping
drop_metadata = ['player', 'team', 'year']
numeric_features = [col for col in X_train_hist.select_dtypes(include=np.number).columns if col not in drop_metadata]
categorical_features = [col for col in X_train_hist.select_dtypes(exclude=np.number).columns if col not in drop_metadata]

# Standard pipeline elements
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), 
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")), 
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

full_preprocessor = ColumnTransformer([
    ("num", num_pipeline, numeric_features),
    ("cat", cat_pipeline, categorical_features)
])

# Single Feature Baseline Configuration ('pts' is chosen as our core volume metric)
target_single_feature = 'pts' if 'pts' in X_train_hist.columns else 'mp'
single_feature_preprocessor = ColumnTransformer([
    ("num", num_pipeline, [target_single_feature])
], remainder="drop")

print(f"Pipelines built. Single-feature baseline model will track: '{target_single_feature}'")

Pipelines built. Single-feature baseline model will track: 'pts'


In [16]:
# Dictionary of modeling approaches to track
base_models = {
    "Dummy Heuristic (Mean)": (DummyRegressor(strategy="mean"), full_preprocessor),
    f"Single-Feature Baseline ({target_single_feature})": (LinearRegression(), single_feature_preprocessor),
    "Linear Regression (OLS)": (LinearRegression(), full_preprocessor),
    "Ridge Regression": (Ridge(alpha=1.0), full_preprocessor),
    "Decision Tree Regressor": (DecisionTreeRegressor(max_depth=5, random_state=26), full_preprocessor),
    "Random Forest Regressor": (RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=26), full_preprocessor)
}

model_cv_summary = []

# Chronological Validation Loop
for name, (model, preprocessor) in base_models.items():
    fold_rmses = []
    fold_mapes = []
    fold_cpws_maes = []
    
    # Generate folds dynamically using structural splitter
    fold_generator = splitter.generate_historical_cv_folds(X_train_hist, y_train_hist)
    
    for fold, (train_idx, val_idx) in enumerate(fold_generator, start=1):
        X_tr, y_tr = X_train_hist.iloc[train_idx], y_train_hist.iloc[train_idx]
        X_val, y_val = X_train_hist.iloc[val_idx], y_train_hist.iloc[val_idx]
        
        # Build evaluation pipeline specific to model variant
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("regressor", model)
        ])
        
        # Fit and predict on chronological slice
        pipeline.fit(X_tr, y_tr)
        y_pred = pipeline.predict(X_val)
        
        # Primary KPI: RMSE
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        fold_rmses.append(rmse)
        
        # Secondary KPI: MAPE
        mape = mean_absolute_percentage_error(y_val, y_pred)
        fold_mapes.append(mape)
        
        # Business ROI KPI: CPWS MAE
        # Map structural identifiers back to compute valuation efficiency metrics by team
        val_context = X_val.copy()
        val_context['actual_salary'] = y_val.values
        val_context['pred_salary'] = y_pred
        
        # Safeguard 'ws' (Win Shares) against 0 value to eliminate division anomalies
        val_context['ws'] = val_context['ws'].astype(float)
        
        team_aggregates = val_context.groupby('team').agg({
            'actual_salary': 'sum',
            'pred_salary': 'sum',
            'ws': 'sum'
        })
        
        # Drop teams with 0 or negative aggregate win shares to guarantee valid benchmarks
        team_aggregates = team_aggregates[team_aggregates['ws'] > 0]
        
        actual_cpws = team_aggregates['actual_salary'] / team_aggregates['ws']
        pred_cpws = team_aggregates['pred_salary'] / team_aggregates['ws']
        
        cpws_mae = (actual_cpws - pred_cpws).abs().mean()
        fold_cpws_maes.append(cpws_mae)
        
    # Standardize cross-validation scores via mean aggregation
    model_cv_summary.append({
        "Model Architecture": name,
        "Primary KPI: RMSE": np.mean(fold_rmses),
        "Secondary KPI: MAPE": f"{np.mean(fold_mapes) * 100:.2f}%",
        "Business KPI: CPWS MAE": f"${np.mean(fold_cpws_maes):,.2f} per WS"
    })

# Format results for visual analytics audit
results_df = pd.DataFrame(model_cv_summary)

In [17]:
print("CV Baseline Performance")
print(results_df.to_string(index=False))

CV Baseline Performance
           Model Architecture  Primary KPI: RMSE      Secondary KPI: MAPE Business KPI: CPWS MAE
       Dummy Heuristic (Mean)       69597.131729 73705670780178161664.00%     $141,172.69 per WS
Single-Feature Baseline (pts)       48794.201500 33841901106600939520.00%      $69,277.75 per WS
      Linear Regression (OLS)       52720.092685 44973254607995723776.00%      $34,313.41 per WS
             Ridge Regression       49780.306324 52534922533414756352.00%      $36,609.21 per WS
      Decision Tree Regressor       56171.981342 70213000531146113024.00%      $55,335.94 per WS
      Random Forest Regressor       44850.440577 42333535659496890368.00%      $25,573.41 per WS
